In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# INPUTS and constants:
t_f_dynamic = np.array([...]) # Time of fall voltage off
t_r_dynamic = np.array([...]) # Time of rise with voltage on
V_dynamic = np.array([...]) # Applied voltage used to make droplet rise

t_f_balancing = np.array([...]) # Time of fall with voltage off
V_balancing = np.array([...]) # Voltage required to hold droplet stationary

L = 1e-3 # Distance between top and bottom lines in m
d = 5e-3 # Distance between capacitor plates in m
g = 9.81 # Acceleration due to gravity
rho_air = 1.2 # density of air in kg/m^3
rho_oil = 928 # density of oil in kg/m^3
eta = 1.8432e-5 # viscosity of air in SI units
b = 6.17e-8 # Cunningham correction in SI units
P = 0.76 # m of Hg
e = 1.602e-19

delta_L = 0
delta_t = 0.1
delta_d = 0
delta_V = 0.1

def calculate_velocity(time_array, distance):
    """
    Calculates the velocity lol
    """ 

    return distance / time_array

def calculate_radius(v_f):
    """ 
    Calculates the radius of the droplet using quadratic formula
    """

    A = 1
    B = b / P
    C = - (9 * eta * v_f) / (2 * g * (rho_oil - rho_air))

    radius = (-B + np.sqrt(B**2 - 4 * A * C)) / (2 * A)
    
    return radius

def calculate_balancing_method(t_f_array, V_b_array):
    """ 
    Calcualtes the total charge and its absolute uncertainty
    """

    v_f = calculate_velocity(t_f_array, L)
    r = calculate_radius (v_f)

    q = (4/3) * np.pi * (rho_oil - rho_air) * g * (r**3) * d / V_b_array

    frac_error_vf = np.sqrt((delta_L / L)**2 + (delta_t / t_f_array)**2)
    
    # Fractional error in charge
    frac_error_q = np.sqrt( (9/4) * (frac_error_vf**2) + 
                            (delta_d / d)**2 + 
                            (delta_V / V_b_array)**2 )
    
    # Absolute error in charge
    delta_q = q * frac_error_q
    
    return q, delta_q

def calculate_dynamic_method (t_f_array, t_r_array, V_array):
    """
    Calcualtes the total charge and its absolute uncertainty 
    """

    v_f = calculate_velocity(t_f_array, L)
    v_r = calculate_velocity(t_r_array, L)
    r = calculate_radius(v_f)   

    q = (6 * np.pi * eta * d * r * (v_f + v_r)) / V_array

    frac_error_vf = np.sqrt((delta_L / L)**2 + (delta_t / t_f_array)**2)
    frac_error_vr = np.sqrt((delta_L / L)**2 + (delta_t / t_r_array)**2)

    abs_error_vf = v_f * frac_error_vf
    abs_error_vr = v_r * frac_error_vr

    sum_term_error = (abs_error_vf**2 + abs_error_vr**2) / ((v_f + v_r)**2)

    frac_error_q = np.sqrt( (delta_d / d)**2 + 
                            (delta_V / V_array)**2 + 
                            (1/4) * (frac_error_vf**2) + 
                            sum_term_error )
    
    delta_q = q * frac_error_q
    
    return q, delta_q

def quantisation (q_array, delta_q_array):
    """
    Decodes total charge into integer n and individual e values.
    """

    # We round the ratio of measured charge to theoretical charge
    n_array = np.round(np.abs(q_array) / e)
    n_array[n_array == 0] = 1 # Safety check for very small q
    
    # Calculate the experimental charge of a single electron (e = q/n)
    e_exp_array = np.abs(q_array) / n_array
    
    # Propagate the error for each individual e (delta_e = delta_q / n)
    delta_e_array = delta_q_array / n_array
    
    return n_array.astype(int), e_exp_array, delta_e_array


# Choose your method and run (e.g., Balancing)
q_final, dq_final = calculate_balancing_method(t_f_balancing, V_balancing)

n_list, e_list, de_list = quantisation(q_final, dq_final)

print(f"Total Charges (q): {np.abs(q_final)}")
print(f"Charge Errors (dq): {dq_final}")
print(f"Integer counts (n): {n_list}")
print(f"Electron charge (e): {e_list}")
print(f"Electron errors (de): {de_list}")

e_avg = np.mean(e_list)
e_sem = np.std(e_list, ddof=1) / np.sqrt(len(e_list))
print(f"FINAL RESULT: e = {e_avg:.4e} +/- {e_sem:.2e} C")